# Airport Network Resilience to Major Hub Disruptions

### Summary
If an Air Lines closes one of it's major hubs, how many stranded passengers can be accommodated elsewhere, and what interventions would help the most?. The goal of this study is to measure how much passenger traffic a U.S. airport network can carry when a major hub is temporarily closed. This could happen for many reasons including government shutdowns, TSA workers go without pay, air traffic control communication outage or extreme weather.
The data used in this study comes from the Bureau of Transportation Statistics <sup>[1](#ref1)</sup>. We focused only on regular passengers (Class F) operated by Delta Air Lines (DL) and it's subsidiary Endeavor Air (9E) in 2025. To answer this question, the maximum network flow will give us a good insight on the airport network resilience faced to a major hub closure. We model the air line network as a graph in which airports are nodes and the flights are the arcs connecting a departure node and a destination node. Each arc has a capacity that is equal to the average number available seats per day. Computing the maximum flow will indicate how many stranded passengers can be rebooked onto alternative routes. We will also identify the bottleneck flights using Max-flow min-cut theorem; in other words, we reveal the saturated fights that will limit the maximum flow of people. We finally conduct a sensitivity analyses and explore different strategies for improving the maximum flow on the network. Because the model ignores individual passenger destinations and connection timing, the results represent a best-case limit. 

---
<a id="ref1"></a>[1] Bureau of Transportation Statistics, [T-100 Domestic Segment data](https://transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=GEE&QO_fu146_anzr=Nv4%20Pn44vr45)

### Data preparation

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import time
df = pd.read_csv("../data/raw/T_T100D_SEGMENT_ALL_CARRIER.csv")
df = df.sort_values("PASSENGERS",ascending=False)
df.head(15)

Let's filter the data and calculate the stats per day

In [ ]:
#Keep only the delta flights in this study Detla Air and Endeavor Air Lines
delta = df[df["UNIQUE_CARRIER"].isin(["DL","9E"])]
delta = delta[delta["CLASS"] =="F"]
#Remove flights that never flew
delta = delta[delta["SEATS"]>0]
#Cleanup records where flights returned to the same airport
delta = delta[delta["ORIGIN"]!= delta["DEST"]]
#Calculate the capacity of each arc (route) by collapsing all the rows for each (Origin,Destintion) pair into a single row then calculate the avereage for each group.
routes = (
    delta.groupby(["ORIGIN","DEST"],as_index=False)
         .agg(SEATS_YEAR =("SEATS","sum"),
              PASSENGERS_YEAR=("PASSENGERS","sum"), 
              MEAN_DISTANCE=("DISTANCE","mean")))

#Convert yearly totals to an average day. Empty seats per day = rebooking capacity of the route
routes["SEATS_PER_DAY"] = routes["SEATS_YEAR"] / 365
routes["PASSENGERS_PER_DAY"] = routes["PASSENGERS_YEAR"] / 365
routes["EMPTY_PER_DAY"] = routes["SEATS_PER_DAY"] - routes["PASSENGERS_PER_DAY"] 
# Only consider flights that have empty seats
routes = routes[routes["EMPTY_PER_DAY"] > 0]
routes.head(10)

### Hub Selection
First we need to identify the airport that has the most siginificant passengers traffic including in transit or not in transit passengers. It will be a good candidate as a major hub for this study.

In [ ]:
# "seats_out" counts seats available on flights leaving from each airport
# "seats_in" counts seats available on flights arriving at each airport
# "hub_size" departing seats plus arriving seats for each airport
seats_out = routes.groupby("ORIGIN")["SEATS_PER_DAY"].sum()
seats_in  = routes.groupby("DEST")["SEATS_PER_DAY"].sum()
hub_size  = (seats_out.add(seats_in, fill_value=0)).sort_values(ascending=False)
hub_size.head(10)


ATL is the airport with the largest total daily seat capacity on it's flights counting both departure and arrival. Let's calculate it's share.

In [ ]:
atl = routes[(routes["ORIGIN"] == "ATL") | (routes["DEST"] == "ATL")]
print(f"Share of Delta seats touching ATL: {atl['SEATS_YEAR'].sum() / routes['SEATS_YEAR'].sum():.1%}")

### Linear Program Formulation

To formulate the maximum flow problem as a linear programming (LP) model, let $G = (V, E)$ be a directed graph where $V$ is the set of vertices and $E$ is the set of arcs. Let $s \in V$ be the source node, and $t \in V$ be the sink node. For each arc $e \in E$, 

$$
\left\{
\begin{align}
&\text{maximize} \quad \sum_{e \in \text{out}(s)} f(e) & (1)\\[1.5ex]
&\text{subject to} \quad \sum_{e \in \text{in}(v)} f(e) - \sum_{e \in \text{out}(v)} f(e) = 0, \quad \forall v \in V \setminus \{s, t\} & (2)\\[1ex]
&\phantom{\text{subject to}} \quad f(e) \le C(e), \quad \forall e \in E & (3)\\[1ex]
&\phantom{\text{subject to}} \quad f(e) \ge 0, \quad \forall e \in E & (4)
\end{align}
\right.
$$
The objective function $(1)$ maximizes the total amount of flow coming out of the source node $S$. The constraints $(2)$ enforce flow conservation, for all nodes other than the source $S$ and the sink $T$. The constraints $(3)$ define the arcs capacities restricting the amount of flow that can go through each edge. The constraints $(4)$ define the flow variables specifying the amount of flow assigned to each arc must be positive. 

Although solvable in polynomial time via generic linear programming (LP) formulations, the maximum flow problem is more efficiently addressed using a dedicated network flow algorithm optimized for graph structures.

### Methodology

The maximum-flow problem is defined on a network made of nodes and arcs, where each arc links a pair of nodes and has a capacity. It requires a source node $S$, where the flow starts, and a sink node $T$, where the flow ends. Several algorithms can be used to solve this problem, including the Ford–Fulkerson algorithm. The algorithm repeatedly increases the flow from the source to the sink along paths with available residual capacity. This process continues until no path remains through which additional flow can be sent. These saturated arcs form the minimum-cut.

In this study, we use the maximum-flow algorithm to find the capacity of Delta Air Lines’ network when a hub is closed. To do so, we replace the hub node with a source node and a sink node. The source node sends a total flow equal to the number of passengers that belong to flights going to the closed hub. The sink node receives the passengers who were expected at the destinations out of the closed hub. The maximum flow is therefore the largest number of stranded passengers who can reach their destinations using alternative routes. Note that ATL node will have the largest weighted degree in the directed graph (sums of the numerical weights of those connected edges) as it has the largest total daily seat capacity.

Below is an illustration of how we substitue the closed hub node with a source node $S$ and a sink node $T$. The graph on the left shows a small network before the Atlanta hub (ATL) is closed. The graph o the right shows the same network after the ATL node is deleted, with a source node $S$ and a sink node $T$ added artificially. 100 are in Boston going to Orlando and 80 are in Orlando going to Boston. Each canceled flight into ATL becomes an arc from $S$ to that flight’s origin airport, and each canceled flight out of ATL becomes an arc from that flight’s destination airport to $T$. In both cases, the arc’s capacity equals the number of passengers on the canceled flight. On this new directed graph, the flow algorithm will search alternative routes for the stranded passengers by searching for maximum fow from $S$ to $T$.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Graph Before: Atlanta is open
before = nx.DiGraph()
before.add_weighted_edges_from([
    ("BOS", "ATL", 100), ("ATL", "MCO", 100),   # 100 passengers BOS -> ATL -> MCO
    ("MCO", "ATL", 80),  ("ATL", "BOS", 80),    # 80 passengers MCO -> ATL -> BOS
    ("BOS", "DTW", 30),  ("DTW", "BOS", 40),    # empty seats
    ("MCO", "DTW", 20),  ("DTW", "MCO", 50),    # empty seats
], weight="capacity")

# Graph After: Atlanta removed, S and T added
after = before.copy()
after.remove_node("ATL")
after.add_weighted_edges_from([
    ("S", "BOS", 100), ("S", "MCO", 80),        # passengers stuck
    ("BOS", "T", 80),  ("MCO", "T", 100),       # passengers expected
], weight="capacity")

# Draw graphs side by side in a grid of charts
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

pos_before = nx.circular_layout(before)
# Display graph 2 in three columns style
pos_after = nx.multipartite_layout(after, subset_key={0: ["S"], 1: ["BOS", "DTW", "MCO"], 2: ["T"]})

for G, pos, ax, title in [(before, pos_before, axes[0], "Before: ATL open"),
                          (after, pos_after, axes[1], "After: ATL removed")]:
    colors = ["#FFBF00" if n in ("S", "T") else "lightblue" for n in G.nodes()]
    nx.draw(G, pos, ax=ax, with_labels=True, node_size=2500,font_size=9,
            node_color=colors, connectionstyle="arc3,rad=0.15")
    nx.draw_networkx_edge_labels(G, pos, nx.get_edge_attributes(G, "capacity"),
                                 ax=ax, connectionstyle="arc3,rad=0.15")
    ax.set_title(title)
plt.show()

### Splitting The Nodes

In addition to creating a source and a sink node it is necessary to split the nodes connected to them into departure and arrival nodes. Below is an illustration of how this is performed. Indeed, on the left graph, Boston is connected to both $S$ and $T$, so the algorithm can see the path $S$ $\rightarrow$ $BOS$ $\rightarrow$ $T$. That counts a passenger as arrived in Boston when they never left Boston and never took an alternative route. The same happens at Orlando, so the left graph reports a flow max at 180, meaning all stranded passenger were rebooked. To address this issue, we split each airport into an arrival node and a departure node. The arc from an airport’s arrival node to its departure node represents passengers connecting at that airport, and it has unlimited capacity.

The directe graph on the right is an illustration of what we will be building, there’s no arrow from $BOS_\text{dep}$ to $BOS_\text{arr}$ . The forces the algorithm to pick an alternative path. Since the shortcut is gone, the max flow drops to 50 passengers, which is all the Detroit flights can carry (30 seats out of Boston + 20 out of Orlando).

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Graph 1 : Single node per airport
single = nx.DiGraph()
single.add_weighted_edges_from([
    ("S", "BOS", 100), ("S", "MCO", 80),          # passengers stuck
    ("BOS", "T", 80),  ("MCO", "T", 100),         # passengers expected
    ("BOS", "DTW", 30), ("DTW", "BOS", 40),       # flights (empty seats)
    ("MCO", "DTW", 20), ("DTW", "MCO", 50),
], weight="capacity")

# Graph 2: Split node for each airport
split = nx.DiGraph()
split.add_weighted_edges_from([
    ("S", "BOS_dep", 100), ("S", "MCO_dep", 80),             # S only goes to departures
    ("BOS_arr", "T", 80),  ("MCO_arr", "T", 100),            # T only comes from arrivals
    ("BOS_dep", "DTW_arr", 30), ("DTW_dep", "BOS_arr", 40),  # flights: dep -> arr
    ("MCO_dep", "DTW_arr", 20), ("DTW_dep", "MCO_arr", 50),
], weight="capacity")
split.add_edges_from([("BOS_arr", "BOS_dep"),               # Split nodes (connections)
                       ("DTW_arr", "DTW_dep"),              # capacity = infinity
                       ("MCO_arr", "MCO_dep")])              

# Display graph 1 in three columns style
pos_single = nx.multipartite_layout(single, subset_key={0: ["S"], 1: ["BOS", "DTW", "MCO"], 2: ["T"]})
# Display graph 2 in four columns style
pos_split = nx.multipartite_layout(split, subset_key={0: ["S"],
                                                        1: ["BOS_dep", "DTW_dep", "MCO_dep"],
                                                        2: ["BOS_arr", "DTW_arr", "MCO_arr"],
                                                        3: ["T"]})

# Draw graphs side by side, with the max flow in each title
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for G, pos, ax, title in [(single, pos_single, axes[0], "One node per airport"),
                          (split, pos_split, axes[1], "Two nodes per airport")]:
    # assign amber color for T and S, lightblue otherwise
    colors = ["#FFBF00" if n in ("S", "T") else "lightblue" for n in G.nodes()]
    #Display "∞" for connection flights, display arc capacity otherwise
    labels ={}
    for u,v,capa in G.edges(data= True):
        labels[(u,v)] = capa.get("capacity","∞")
        
    nx.draw(G, pos, ax=ax, with_labels=True, node_size=2500, font_size=9,
            node_color=colors, connectionstyle="arc3,rad=0.15")
    nx.draw_networkx_edge_labels(G, pos, labels,
                                 ax=ax, connectionstyle="arc3,rad=0.15")
    ax.set_title(f"{title}\nMax flow = {nx.maximum_flow_value(G, 'S', 'T')}")
plt.show()

### Constructing A Directed Graph For The Delta Air Lines Network

The function below constructs a directed graph representing the network without the hub, based on the routes and the proportion of stranded passengers. In this study, we assume that 60% of passengers travelling to Atlanta will be affected.

In [ ]:
HUB = "ATL"
share = 0.6   # 60% assumed share of ATL passengers who are connecting

def GraphBuilder(routes, hub, stranded_share, log=False):  

    # STRANDED connecting passengers stuck at their ORIGIN airport 
    into_hub = routes[routes["DEST"] == hub][["ORIGIN", "PASSENGERS_PER_DAY"]].copy()
    into_hub["STRANDED"] = stranded_share * into_hub["PASSENGERS_PER_DAY"]

    # DEMAND = connecting passengers who want to reach each DESTINATION 
    out_hub = routes[routes["ORIGIN"] == hub][["DEST", "PASSENGERS_PER_DAY"]].copy()
    out_hub["DEMAND"] = stranded_share * out_hub["PASSENGERS_PER_DAY"]
  
    flights = routes[(routes["ORIGIN"] != hub) & (routes["DEST"] != hub)]
    
    G = nx.DiGraph()
    # Flights: departure node to arrival node, capacity = empty seats per day
    for o, d, cap in zip(flights["ORIGIN"], flights["DEST"], flights["EMPTY_PER_DAY"]):
        G.add_edge(o + "_dep", d + "_arr", capacity=cap)
    
    # Connections: arrival to departure at the same airport. Unlimited passengers are authorized
    airports = set(routes["ORIGIN"]) | set(routes["DEST"])
    airports.discard(HUB)     # discard the closed Hub
    for a in airports:
        G.add_edge(a + "_arr", a + "_dep") 
    
    # Super source to origins, capacity = stranded passengers
    for a, s in zip(into_hub["ORIGIN"], into_hub["STRANDED"]):
        G.add_edge("S", a + "_dep", capacity=s)
    
    # Destinations to super sink, capacity = demand
    for a, dem in zip(out_hub["DEST"], out_hub["DEMAND"]):
        G.add_edge(a + "_arr", "T", capacity=dem)

    if log:
        print("Flights to Hub:", len(into_hub), "| Stranded per day:", round(into_hub["STRANDED"].sum()))
        print("Flights from Hub:", len(out_hub), "| Demand per day:", round(out_hub["DEMAND"].sum()))
        print("Flights remaining:", len(flights))
        print("Nodes:", G.number_of_nodes(), "| Arcs:", G.number_of_edges())
        
    return G, into_hub, out_hub

G, into_hub, out_hub = GraphBuilder(routes,HUB,share,True)

### Stranded Passengers With No Alternative Route

Let’s identify the number of passengers that can reach their destination only thougth Atlanta. With the hub closed, these passengers have no alternative route, so they will remain stranded until Atlanta reopens. To do so, we look for every direct flight to ATL and check whether its departure airport is not connected to any other one. This can be done using a graph property called the out-degree of a node, which is the number of arcs leaving that node. A node with an out-degree of zero has no outgoing arcs.

In [ ]:
to_hub_only = [a for a in into_hub["ORIGIN"] if G.out_degree(a + "_dep") == 0]
num_stuck = into_hub[into_hub["ORIGIN"].isin(to_hub_only)]["STRANDED"].sum()
print(f"Airports with no Delta flight except to ATL: {to_hub_only}")
print(f"{num_stuck:,.0f} passengers per day have no rebooking option.")

### Solving the Maximum Flow Problem

In [ ]:
def FlowMaxSolver(G,log=False):
    timer_start = time.perf_counter()
    max_flow, flow = nx.maximum_flow(G, "S", "T")
    timer_end = time.perf_counter()
    elapsed =(timer_end - timer_start)
    if log:
        print(f"Solver elapsed time: {elapsed:,.1f} s")
    return (max_flow, flow)
    
max_flow, flow = FlowMaxSolver(G,True)
total = into_hub["STRANDED"].sum()
print(f"Rebooked per day: {max_flow:,.0f} passengers of {total:,.0f} - rebooked {max_flow/total:.1%}")

The optimal solution finds alternative routes for 49.7% of the stranded passengers. No all passengers reached their destination and that could be explained because some flights (arcs) in the network reach their full capacity and prevent the rest of the flow moving from the source node to the sink node.

### Sensitivy Analyses

Let's perform a sensitivity analyses to assess how changes in the share parameter affect the maximum flow in the network. We will build different graphs and run the flow algoirthm for stranded share values ranging from 10% to 90%

In [ ]:
def run_sensitivity(HUB):
    routes_sens = routes.copy()
    rows_sens = []
    for share in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8,0.9]:
        G_sens, into_hub_sens, out_hub_sens = GraphBuilder(routes_sens,HUB,share)
        max_flow_sens, flow_sens = FlowMaxSolver(G_sens)
        total_sens = into_hub_sens["STRANDED"].sum()
        rows_sens.append({"share%":share*100,
                          "total stranded":total_sens,
                          "rebooked": max_flow_sens,
                          "rebooked%":(max_flow_sens/total_sens)*100,
                          "still stranded": total_sens - max_flow_sens})
    
    return pd.DataFrame(rows_sens)
        
sens_share = run_sensitivity(HUB)

In [ ]:
fig,(ax1,ax2)= plt.subplots(
    1,2,
    figsize=(16,6),
    gridspec_kw={"width_ratios":[0.8,1]}
)

# Dataframe
ax1.axis("off")
table = ax1.table(
        cellText= sens_share.round(2).values,
        colLabels = sens_share.columns,
        loc = "center",
        cellLoc="center"
)
table.auto_set_font_size("False")
table.set_fontsize(12)
table.scale(2,2)
ax1.set_title("Sensitivy Analyses Data")
for (row, col), cell in table.get_celld().items():
    if row==0:
        cell.set_text_props(weight="bold")
# Charts

# Rebooked
ax2.plot(
    sens_share["share%"],
    sens_share["rebooked"],
    marker="o",
    label="Rebooked"
)
# Still stranded
ax2.plot(
    sens_share["share%"],
    sens_share["still stranded"],
    marker="o",
    label="Still Stranded"
)
# Plot
ax2.set_xlabel("Share %")
ax2.set_ylabel("Number of Passengers")
ax2.set_title("Rebooked vs. Still Stranded by Share")
ax2.set_xticks(sens_share["share%"])
ax2.legend()
plt.tight_layout()
plt.show()

Rebooked passengers grow from 7870 to 33705 as the share goes from 10% to 90%, but the rebooked percentage falls. If 90% of Atlanta passengers were connecting, the network could absorb roughly 33705 a day out of 86372. The network saturates as the share rises. This can be explained by the max-flow min-cut theorem as the maximum flow equals the capacity of the minimum cut, and it is mostly fixed because the capacities of the intermediate flight arcs remain unchanged. They are the fixed part of the network.

### The Bottleneck Flights 

Let’s find the flights that create a bottleneck and prevent all stranded passengers from reaching their destinations. This can be done by computing the minimum cut of the graph. A cut divides the nodes into two sets, one containing the source $S$ and the other containing the sink $T$. The minimum cut is the separation for which the sum of the capacities of the arcs crossing it is the smallest. By the max-flow min-cut theorem, this total equals the maximum flow, and the arcs crossing the cut are the saturated flights that limit it.

In [ ]:
cut_value, (reachable, non_reachable) = nx.minimum_cut(G, "S", "T")
# Collect a list of tuples (arc, capacity) that have the departure node is the source side of the cut, 
# and the destination node in the sink side of the cut
cut_arcs = []
for u in reachable:
    for v in G[u]: # v is successor of u
        if v in non_reachable:
            cap = G[u][v]["capacity"]
            f = flow[u][v]
            residual = cap - f
            cut_arcs.append((u,v,cap,f,residual))
            
print("Min cut value:", round(cut_value), "| Arcs in cut:", len(cut_arcs))

# Filter out artificial fights going out from S or going in T
def is_flight(x):
    return x[0] != "S" and x[1] != "T"
    
bottleneck_flight = sorted([x for x in cut_arcs if is_flight(x)], key=lambda x: x[2],reverse=True)
bottleneck_df = pd.DataFrame([x for x in bottleneck_flight], 
                            columns=["from","to","capacity","flow","residual"])
print("Top 10 Flights causing the bottleneck:")
bottleneck_df.head(10)

### Strategies for Improving Network Flow

####  A-Increasing Empty Seats on One of the Minimum-Cut Flights

Let’s add empty seats to one of the cut arcs to try to increase the maximum flow. For example, we can pick the LAX $\rightarrow$ AUS flight and add an empty Airbus A350-900 with 306 seats.

In [ ]:
routes_upg = routes.copy()
row_upg = (routes_upg["ORIGIN"] == "LAX") & (routes_upg["DEST"] == "AUS")
routes_upg.loc[row_upg, "EMPTY_PER_DAY"] += 306

G2,into_hub_upg, _ = GraphBuilder(routes_upg,HUB,share,False)
max_flow_upg,_ = FlowMaxSolver(G2,False)

total_upg = into_hub_upg["STRANDED"].sum()
print(f"Rebooked per day: {max_flow_upg:,.0f} passengers of {total_upg:,.0f} - rebooked {max_flow_upg/total_upg:.1%}")

Increasing the empty seats on the LAX $\rightarrow$  AUS flight did not significantly improve the total flow: only 0.2% additional passengers reached their destination. The LAX → AUS arc is in the minimum cut, so adding 306 seats to that flight let more passengers fly. However another arc on the same path filled up and became part of the new minimum cut preventing the rest of the stranded passengers to reach their destinations.

Adding more capacity to one route does not always mean that more passengers can be rerouted across the entire network. Even if the upgraded route can carry more passengers, other routes may still limit the total number of passengers who can be rerouted.

####  B-Creating A Temporary Hub

Instead of increasing the capacity of the minimum cut arcs, we try this time to increase the maximum flow by creating new flights through a temporary hub. In this example, Detroit (DTW) will serve as the temporary hub and suppose it will handle 20% of Atlanta’s seats. We add arcs from the airports where passengers are stranded to DTW, and from DTW to the destinations.

In [ ]:
NEW_HUB = "DTW"   # temporary hub
SHARE = 0.2       # share of Atlanta's seats

#Helper function that creates additional seats/flights
def add_seats(graph, u, v, seats):
    if graph.has_edge(u, v):
        graph[u][v]["capacity"] += seats
    else:
        graph.add_edge(u, v, capacity=seats)

G_Tmp_Hub = G.copy()

#Creating new arcs from stranded passengers’ airports to the new hub
to_atl = routes[routes["DEST"] == HUB]     

for index,row in to_atl.iterrows():
    origine = row["ORIGIN"]
    seats = row["SEATS_PER_DAY"]
    if origine != NEW_HUB:
        add_seats(G_Tmp_Hub, origine + "_dep", NEW_HUB + "_arr", SHARE * seats)  
        
#Creating new arcs from new hub to stranded passengers’ airports 
from_atl = routes[routes["ORIGIN"] == HUB] 

for _,row in from_atl.iterrows():
    destination = row["DEST"]
    seats = row["SEATS_PER_DAY"]
    if destination != NEW_HUB:
        add_seats(G_Tmp_Hub, NEW_HUB + "_dep", destination + "_arr", SHARE * seats)  
        
# Run MaxFlow
new_flow_Tmp_Hub = nx.maximum_flow_value(G_Tmp_Hub, "S", "T")

print(f"Rebooked per day {new_flow_Tmp_Hub:,.0f} - additional rerouted {new_flow_Tmp_Hub - max_flow:,.0f}")

Assigning a temporary hub can significantly help maximize the flow on the network. Using this approach, there is over 15,899 additional passengers that can reach their destination. Now there are new paths from $S$ to $T$ that bypasses the old saturated arcs and they are no longer a barrier.

## Conclusion

If the Atlanta hub closes, Delta’s network can reroute at most 28,633 of the 57,582 stranded passengers per day (49.7%), based on average 2025 data. Because the flow network ignores individual passenger destinations and flight schedules, the solution is an upper bound. The rest of the passengers are limited by saturated flights, identified by the minimum cut arcs, or by airports connected to the network only through Atlanta. These flights are where temporary measures, such as larger aircraft or extra flights, would reroute the passengers. Adding a temporary hub can improve considerably more maximum flow than upgrading individual flights, but this depends on the airport being able to handle the extra traffic.

## Author
 
<a href="https://linkedin.com/in/nazimcoindet" target="_blank">Nazim Coindet</a>

**Date** October 19, 2026
